In [17]:
print("Cell 1")

import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Try multiple API keys in order of preference (OPENROUTER first)
api_keys = {
    "OPENROUTER": os.getenv("OPENROUTER_API_KEY"),
    "GOOGLE": os.getenv("GOOGLE_API_KEY"),
    "GROQ": os.getenv("GROQ_API_KEY")
}

# Find the first available API key
selected_provider = None
api_key = None

for provider, key in api_keys.items():
    if key:
        selected_provider = provider
        api_key = key
        print(f"Using {provider} API key")
        break

if not api_key:
    raise ValueError("No API key found in .env file (tried OPENROUTER, GOOGLE, GROQ)")

print("API Key Loaded Successfully!")

Cell 1
Using OPENROUTER API key
API Key Loaded Successfully!


STATE BACKEND

In [18]:
print("Cell 2")

from deepagents import create_deep_agent
from deepagents.backends import StateBackend

# Import all possible LLM providers
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

Cell 2


In [19]:
print("Cell 3")

# Create the LLM based on selected provider
if selected_provider == "GROQ":
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=api_key
    )
elif selected_provider == "GOOGLE":
    llm = ChatGoogleGenerativeAI(
        model="gemini-pro",
        api_key=api_key
    )
elif selected_provider == "OPENROUTER":
    llm = ChatOpenAI(
        model="openai/gpt-4o-mini",
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1"
    )
else:
    raise ValueError(f"Unknown provider: {selected_provider}")

# Create the DeepAgent
agent = create_deep_agent(model=llm)

Cell 3


In [20]:
print("Cell 4")

# Explicitly providing a backend (equivalent behavior)
agent2 = create_deep_agent(
    model=llm,
    backend=StateBackend()
)

Cell 4


In [21]:
print("Cell 5")

"""
Invoke the agent and ask it to write a file.
StateBackend keeps that file inside the LangGraph state.
"""

result = agent2.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Create a file at /note/todo.txt with exactly this content:\n\n"
                    "1. Record video\n"
                    "2. Edit video\n"
                    "3. Upload video\n\n"
                    "Then tell me you did it."
                ),
            }
        ]
    }
)

# The agent's final natural-language reply
print("\n----------- Agent Reply ---------------------")
print(result["messages"][-1].content)

Cell 5

----------- Agent Reply ---------------------
The file at `/note/todo.txt` has been created with the specified content:

```
1. Record video
2. Edit video
3. Upload video
```


In [22]:
print("cell 6")
result

cell 6


{'messages': [HumanMessage(content='Create a file at /note/todo.txt with exactly this content:\n\n1. Record video\n2. Edit video\n3. Upload video\n\nThen tell me you did it.', additional_kwargs={}, response_metadata={}, id='8e5da827-ccfa-4361-b1bf-549a367a2195'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 5345, 'total_tokens': 5381, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.00082335, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.00082335, 'upstream_inference_prompt_cost': 0.00080175, 'upstream_inference_completions_cost': 2.16e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_dd933fdd28', 'i

In [23]:
print("Cell 7")

# Check whether the StateBackend is working.
# Files written by the agent should appear under result["files"].

print("\n----------- Backend Check ---------------------")

files = result.get("files", {})

if files:
    print(f"StateBackend is working - {len(files)} file(s) in state:\n")

    for path, content in files.items():
        print(f"{path}")
        print("-" * 40)
        print(content)
        print()

else:
    print(
        "No files found in state. Either the agent didn't write a file, "
        "or the backend isn't wired up correctly."
    )

Cell 7

----------- Backend Check ---------------------
StateBackend is working - 1 file(s) in state:

/note/todo.txt
----------------------------------------
{'content': '1. Record video\n2. Edit video\n3. Upload video', 'encoding': 'utf-8', 'created_at': '2026-07-06T08:02:09.086227+00:00', 'modified_at': '2026-07-06T08:02:09.086227+00:00'}



In [24]:
print("Cell 8")

"""
Prove persistence within the same thread:
Feed the returned state back in and ask the agent to read the file.
"""

followup = agent2.invoke(
    {
        "messages": result["messages"] + [
            {
                "role": "user",
                "content": "Read /note/todo.txt back to me.",
            }
        ],
        "files": result.get("files", {}),
    }
)

print("\n----------------- Read Back (Same Thread) -----------------")
print(followup["messages"][-1].content)

Cell 8

----------------- Read Back (Same Thread) -----------------
The content of `/note/todo.txt` is:

```
1. Record video
2. Edit video
3. Upload video
```


#### FileSystemBackend(local Disk)

In [25]:
import deepagents

print(deepagents.__version__)

0.6.12


In [31]:
print("Cell Filesystem 1")

"""
Create the agent with a real disk backend.

Key Parameters:
- root_dir: The directory on disk where the agent's files will be stored. 
  Setting it to the project root ensures files are created outside deepAgent folder.
"""

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

# Set root to project parent directory (outside deepAgent)
ROOT = "/home/moeen/projects/AgenticAI"

# Ensure your 'llm' object is initialized before running this cell
agent_fs = create_deep_agent(
    model=llm,
    backend=FilesystemBackend(
        root_dir=ROOT
    )
)

print(f"Filesystem root: {ROOT}")
print("Filesystem backend initialized successfully.")

Cell Filesystem 1
Filesystem root: /home/moeen/projects/AgenticAI
Filesystem backend initialized successfully.


/tmp/ipykernel_55738/3076017497.py:20: LangChainDeprecationWarning: `FilesystemBackend` `virtual_mode` default will change in deepagents==0.6.0; please specify `virtual_mode` explicitly. Note: `virtual_mode` is for virtual path semantics (e.g., `CompositeBackend` routing) and optional path-based guardrails; it does not provide sandboxing or process isolation. Security note: leaving `virtual_mode=False` allows absolute paths and `'..'` to bypass `root_dir`. Consult the API reference for details.
  backend=FilesystemBackend(


In [ ]:
print("Cell Filesystem 2")

"""
Ask the agent to create a real file on disk.

This demonstrates the agent's built-in file operations safely within the workspace root.
"""

result_fs = agent_fs.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Create a file at /home/moeen/projects/AgenticAI/notes/todo.txt with exactly this content:\n\n"
                    "1. Record video\n"
                    "2. Edit video\n"
                    "3. Upload video\n\n"
                    "Then tell me you did it."
                ),
            }
        ]
    }
)

print("\n---------- Agent Reply -----------------------")
print(result_fs["messages"][-1].content)

Cell Filesystem 2

---------- Agent Reply -----------------------
The file has been created at `/home/moeen/projects/AgenticAI/notes/todo.txt` with the specified content.


In [34]:
print("Cell Filesystem 3")

"""
Check if the DeepAgent filesystem backend actually created files
on the real disk.
"""

from pathlib import Path

ROOT = "/home/moeen/projects/AgenticAI"

print("\n-------------- Backend Check (Real Filesystem) --------------")

# Correct path to the file
disk_path = Path(ROOT) / "notes" / "todo.txt"

# Check if file exists
if disk_path.exists():
    print("✅ Filesystem backend is working - file exists on disk!\n")
    print(f"📁 Path: {disk_path.resolve()}")
    print("-" * 40)

    # Read file content (correct way)
    print("\n📄 File Content:\n")
    print(disk_path.read_text())

else:
    print("❌ File not found on disk.")
    print("👉 Either agent didn't create it or path is wrong.")

Cell Filesystem 3

-------------- Backend Check (Real Filesystem) --------------
✅ Filesystem backend is working - file exists on disk!

📁 Path: /home/moeen/projects/AgenticAI/notes/todo.txt
----------------------------------------

📄 File Content:

1. Record video
2. Edit video
3. Upload video


In [35]:
print("Cell Filesystem 4")

"""
Force the agent to create a file using an explicit full path.
This is for learning/debugging (not recommended in production agents).
"""

result_fs2 = agent_fs.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Create a file at /home/moeen/projects/AgenticAI/notes/todo.txt "
                    "with exactly this content:\n\n"
                    "1. Record video\n"
                    "2. Edit video\n"
                    "3. Upload video\n\n"
                    "Then confirm you created it successfully."
                ),
            }
        ]
    }
)

print("\n---------- Agent Reply -----------------------")
print(result_fs2["messages"][-1].content)

Cell Filesystem 4

---------- Agent Reply -----------------------
The file at `/home/moeen/projects/AgenticAI/notes/todo.txt` has been successfully created with the specified content:

```
1. Record video
2. Edit video
3. Upload video
```


# Deep Agent -- StoreBackend verification

creat a deep agent backed by a langgraph store,Invokes it to write a file on one thread,then proves the backend works by reading that file backe on a Different thread- something StateBacked   cannot do 

In [ ]:
print("cell Storebackend 1")

from langgraph.store.memory import InMemoryStore
from deepagents.backends import StateBackend
store=InMemoryStore()

agent=create_deep_agent(
    model=llm,
    backend=StateBackend(
        #local dev: static namespcae . No deployment runtime needed,
        #in a langsmit deployment you 'd use the user-identity version
        namespace=lambda rt:("demo-user")

    ),
    store=store


)
print("agent create with sotreBackend+static namespace")


In [ ]:
print("Cell StateBackend 2")

import os 
import uuid

# THREAD 1 - write file
thread1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result1 = agent2.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Create a file at /note/todo.txt with exactly this content:\n\n"
                    "1. Record video\n"
                    "2. Edit video\n"
                    "3. Upload video\n\n"
                    "Then tell me you have done it."
                ),
            }
        ]
    },
    config=thread1
)

print("--------------- Agent Reply (Thread 1) =============")
print(result1["messages"][-1].content)

In [ ]:
print("Cell StateBackend 3")

# THREAD 2 - read back on a different thread
thread2 = {"configurable": {"thread_id": str(uuid.uuid4())}}

# This will fail because StateBackend doesn't persist across different threads
# Each thread has its own isolated state
followup = agent2.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Read /note/todo.txt back to me.",
            }
        ]
    },
    config=thread2
)

print("--------------- Agent Reply (Thread 2 - Different Thread) =============")
print(followup["messages"][-1].content)